# Freight assessment — visualize all topics
Upload the four raw kit CSVs to `/content`. The selected policy fills 249 null validation market signals with same-date supplied FEATURE medians. Run the single code cell; it saves four PNG dashboards in `/content/freight_visuals/` and never overwrites CSVs. Reference-imputation family/rolling evidence is labeled separately.


In [ ]:
# FREIGHT ASSESSMENT — ONE-CELL VISUAL DASHBOARD
# Read-only for all supplied/submitted CSVs. Saves dashboard PNGs to freight_visuals/.
# Requires: numpy, pandas, matplotlib, scikit-learn (preinstalled in most Colab runtimes).
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from IPython.display import display, Image

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .18,
                     "axes.spines.top": False, "axes.spines.right": False})
ROOTS = [Path('/content'), Path('/content/data'), Path('.'), Path('data')]
SIGNATURE = {'load_id','pickup','delivery','distance','equipment','weight','date',
             'market_index','quote_signal','posted_rate'}

def locate(names, required=(), optional=False):
    for root in ROOTS:
        for name in names:
            p = root/name
            if p.is_file():
                if not required or required.issubset(set(pd.read_csv(p, nrows=0).columns)):
                    return p
    if optional:
        return None
    raise FileNotFoundError(f'No matching freight CSV: {names} (required columns: {required})')

TRAIN = locate(['train-test.csv','train_test.csv'], SIGNATURE)
VAL = locate(['validation.csv'], SIGNATURE-{'posted_rate'})
DEC = locate(['december-chart-inputs.csv','december_chart_inputs.csv'],
             {'pickup','delivery','distance','equipment','weight','date'})
PRED = locate(['validation_predictions.csv','outputs/validation_predictions.csv'],
              {'load_id','predicted_rate'}, optional=True)
DEC_PRED = locate(['december-chart-inputs-filled.csv','outputs/december-chart-inputs.csv'],
                  {'date','predicted_rate'}, optional=True)
CHART = next((p for root in ROOTS for p in
              [root/'scorer_results/candidate_december.png',root/'reports/candidate_december.png']
              if p.is_file()), None)
OUT = Path('/content/freight_visuals') if Path('/content').exists() else Path('freight_visuals')
OUT.mkdir(parents=True, exist_ok=True)
d = pd.read_csv(TRAIN, parse_dates=['date'])
v = pd.read_csv(VAL, parse_dates=['date'])
dec = pd.read_csv(DEC, parse_dates=['date'])
assert len(d) and len(v) and len(dec) and (d.posted_rate > 0).all() and (d.distance > 0).all()
print('Inputs:', TRAIN, VAL, DEC, '| saved prediction CSV:', PRED,
      '| filled December CSV:', DEC_PRED, '| official chart:', CHART)
print('Rows:',len(d),'train;',len(v),'validation;',len(dec),'December chart inputs')
print('Dates:',d.date.min().date(),'→',d.date.max().date(),'; validation:',v.date.min().date(),'→',v.date.max().date())

# Feature matrix + the SAME train-only 6×MAD-sigma cleaning / log-linear model as Cell 10.
T0 = pd.Timestamp('2025-01-01')
def meds(x):
    return {'market': float(x.market_index.median()),
            'quote': float(x.quote_signal.median()),
            'weight': float(pd.to_numeric(x.weight, errors='coerce').abs().median())}
def X(x, m, cities, with_time=True):
    ld = np.log(pd.to_numeric(x.distance).to_numpy(float))
    market = (pd.to_numeric(x.market_index,errors='coerce').fillna(m['market']).to_numpy(float)
              if 'market_index' in x else np.full(len(x),m['market']))
    quote = (pd.to_numeric(x.quote_signal,errors='coerce').fillna(m['quote']).to_numpy(float)
             if 'quote_signal' in x else np.full(len(x),m['quote']))
    weight = np.log(pd.to_numeric(x.weight,errors='coerce').abs().fillna(m['weight']).to_numpy(float))
    parts = [np.ones((len(x),1)), np.column_stack([ld,ld**2,market,quote,weight,
             (x.equipment=='Reefer').to_numpy(float),(x.equipment=='Flatbed').to_numpy(float)])]
    if with_time:
        parts.append(((pd.to_datetime(x.date)-T0).dt.days.to_numpy(float))[:,None])
    k = len(cities)-1; idx = {c:i for i,c in enumerate(cities)}
    for col in ['pickup','delivery']:
        block = np.zeros((len(x),k))
        for row,city in enumerate(x[col].to_numpy()):
            j = idx.get(city)
            if j is None: block[row,:] = 1/k  # unseen city -> mean fitted city effect
            elif j>0: block[row,j-1] = 1
        parts.append(block)
    return np.hstack(parts)
def fit(train):
    m=meds(train); cities=sorted(train.pickup.unique().tolist())
    base=X(train,m,cities,False)[:,:8] # intercept + 7 non-time continuous features
    y=np.log(train.posted_rate.to_numpy(float))
    b0=np.linalg.lstsq(base,y,rcond=None)[0]; resid=y-base@b0
    mad_sigma=1.4826*np.median(np.abs(resid-np.median(resid)))
    keep=np.abs(resid-np.median(resid)) < 6*mad_sigma
    coef=np.linalg.lstsq(X(train.loc[keep],m,cities),y[keep],rcond=None)[0]
    return {'m':m,'cities':cities,'coef':coef,'keep':keep,'sigma':mad_sigma,'ref_resid':resid}
def predict(model,x,source=None):
    x=x.copy()
    if source is not None:
        by_day=source.assign(_day=pd.to_datetime(source.date)).groupby('_day')[
            ['market_index','quote_signal']].median()
        days=pd.to_datetime(x.date)
        for col in ['market_index','quote_signal']:
            daily=days.map(by_day[col]).to_numpy()
            if col in x: x[col]=pd.to_numeric(x[col],errors='coerce').fillna(pd.Series(daily,index=x.index))
            else: x[col]=daily
    return np.clip(np.exp(X(x,model['m'],model['cities'])@model['coef']),25,40000)
def stats(y,p,mask):
    y=np.asarray(y)[mask];p=np.asarray(p)[mask]
    return {'n':len(y),'MAE':np.mean(np.abs(y-p)),
            'MAPE':100*np.mean(np.abs(y-p)/y),
            'R2':1-np.sum((y-p)**2)/np.sum((y-y.mean())**2)}
def finish(fig,name):
    fig.tight_layout(rect=(0,0,1,.97)); fig.savefig(OUT/name,bbox_inches='tight',dpi=140)
    plt.show();plt.close(fig);print('Saved:',OUT/name)

# ────────────────────────────────────────────────────────────────────────────
# PAGE 1 — INPUT DATA, QUALITY, GEOMETRY, TARGET STRUCTURE
# ────────────────────────────────────────────────────────────────────────────
print('\nPAGE 1 — dataset / quality / geography / target')
full=fit(d); mult=np.exp(full['ref_resid']); flagged=~full['keep']
print('Train-only 6×MAD-sigma threshold:',round(6*full['sigma'],5),
      '| retained:',int(full['keep'].sum()),'| flagged:',int(flagged.sum()))
print('Missing (%):', (100*d.isna().mean()).loc[lambda s:s>0].round(3).to_dict())
print('Negative weights:',int((d.weight<0).sum()),'| exact duplicate rows:',int(d.duplicated().sum()))
print('Unique cities:',len(set(d.pickup)|set(d.delivery)),
      '| validation new cities:',len((set(v.pickup)|set(v.delivery))-(set(d.pickup)|set(d.delivery))))
fig,ax=plt.subplots(4,3,figsize=(17,16)); a=ax.ravel()
monthly=d.groupby(d.date.dt.to_period('M')).size();a[0].bar(monthly.index.astype(str),monthly)
a[0].tick_params(axis='x',rotation=50);a[0].set_title('Loads by month')
miss=(100*d.isna().mean()).sort_values(ascending=False).head(10);a[1].barh(miss.index[::-1],miss.values[::-1]);a[1].set_title('Missing by column (%)')
a[2].hist(d.posted_rate,bins=75,color='#2563eb');a[2].set_title('Target: posted_rate ($)')
samp=d.sample(min(len(d),4500),random_state=4)
a[3].scatter(samp.distance,samp.posted_rate,s=3,alpha=.17);a[3].set(xlabel='distance (mi)',ylabel='rate ($)',title='Distance vs rate (sample)')
for eq,g in d.groupby('equipment'):
    a[4].hist((g.posted_rate/g.distance).clip(0,20),bins=65,alpha=.5,label=eq)
a[4].legend(fontsize=8);a[4].set_title('Rate / distance by equipment')
a[5].hist(d.weight.dropna(),bins=75,color='#059669');a[5].axvline(0,c='black',lw=1);a[5].set_title('Weight (including negative values)')
a[6].hist(np.log(mult),bins=115,range=(-2.1,2.1),color='#64748b')
a[6].axvline(-6*full['sigma'],c='red',ls='--');a[6].axvline(6*full['sigma'],c='red',ls='--')
a[6].set_title('Log(actual / reference fit) and 6×MAD sigma')
tmp=pd.DataFrame({'month':d.date.dt.to_period('M').astype(str),'flag':flagged})
fshare=100*tmp.groupby('month').flag.mean();a[7].bar(fshare.index,fshare)
a[7].tick_params(axis='x',rotation=50);a[7].set_title('Flagged target share by month (%)')
for col,color in [('market_index','#2563eb'),('quote_signal','#d97706')]:
    daily=d.groupby('date')[col].median();a[8].plot(daily.index,daily,color=color,label=col,alpha=.75)
a[8].legend();a[8].set_title('Daily medians: market / quote signals')
rad=np.pi/180;la1=d.pickup_lat*rad;la2=d.delivery_lat*rad;dlon=(d.delivery_lon-d.pickup_lon)*rad
h=2*3958.8*np.arcsin(np.sqrt(np.clip(np.sin((la2-la1)/2)**2+
     np.cos(la1)*np.cos(la2)*np.sin(dlon/2)**2,0,1)))
a[9].scatter(h.iloc[samp.index],d.distance.iloc[samp.index],s=3,alpha=.17)
a[9].plot([0,h.max()],[0,h.max()],ls='--',c='red',lw=1)
a[9].set(xlabel='haversine from supplied coordinates (mi)',ylabel='supplied distance (mi)',
         title='Coordinate geometry vs distance')
city_geo=d[['pickup','pickup_lat','pickup_lon']].drop_duplicates('pickup')
a[10].scatter(city_geo.pickup_lon,city_geo.pickup_lat,s=18)
a[10].set(xlabel='given longitude',ylabel='given latitude',title='Supplied pickup-city coordinates')
lanes=d.groupby(['pickup','delivery']).size()
a[11].hist(lanes,bins=50,color='#8b5cf6');a[11].set_yscale('log')
a[11].set(xlabel='rows per pickup→delivery lane',title='Lane recurrence (log y)')
fig.suptitle('01 / Freight assessment — dataset and audit visualizations',fontsize=16)
finish(fig,'01_data_quality.png')

# ────────────────────────────────────────────────────────────────────────────
# PAGE 2 — TEMPORAL HOLDOUT + MODEL FAMILIES (SAME COMPARISON SETTINGS AS AUDIT)
# ────────────────────────────────────────────────────────────────────────────
print('\nPAGE 2 — validation protocols and model comparison')
tr=d[d.date<'2025-09-01'];te=d[d.date>='2025-09-01']
held=fit(tr);y=te.posted_rate.to_numpy(float)
pred_ref=predict(held,te)
pred_selected=predict(held,te,source=te)
clean_mask=np.abs(np.log(y)-np.log(pred_ref))<.4 # test labels ONLY for reporting
assert np.array_equal(clean_mask,np.abs(np.log(y)-np.log(pred_selected))<.4)
score_rows=[]
for policy,preds in [('selected same-date features',pred_selected),
                     ('reference training medians',pred_ref)]:
    for label,mask in [('clean scored rows',clean_mask),('ALL scored rows',np.ones(len(te),bool))]:
        score_rows.append({'policy':policy,'group':label,**stats(y,preds,mask)})
print('Jan–Aug → Sep–Oct holdout (same trained model and clean evaluation mask):')
print(pd.DataFrame(score_rows).round(4).to_string(index=False))
pred=pred_selected # plotted holdout lines/residuals are the selected policy

# Fixed clean mask for FAMILY reporting, independent of any compared family.
# This reproduces the Cell-9 distance-only full-dataset diagnostic; no holdout data is fit
# into any compared model (only the evaluation mask uses observed holdout labels).
ld_all=np.log(d.distance.to_numpy(float));yy=np.log(d.posted_rate.to_numpy(float))
bd=np.linalg.lstsq(np.column_stack([np.ones(len(d)),ld_all]),yy,rcond=None)[0]
mask_family=(np.abs(yy-np.column_stack([np.ones(len(d)),ld_all])@bd)<.4)[d.date>='2025-09-01']
m=meds(tr);cities=sorted(tr.pickup.unique().tolist()); logy=np.log(tr.posted_rate.to_numpy(float))
family={}
family['mean']=np.full(len(te),np.exp(logy.mean()))
B=np.linalg.lstsq(np.column_stack([np.ones(len(tr)),np.log(tr.distance)]),logy,rcond=None)[0]
family['distance']=np.exp(np.column_stack([np.ones(len(te)),np.log(te.distance)])@B)
for with_time in [False,True]:
    xtr=X(tr,m,cities,with_time);xte=X(te,m,cities,with_time)
    b=np.linalg.lstsq(xtr,logy,rcond=None)[0]
    family['linear + time' if with_time else 'linear no time']=np.exp(xte@b)
    t=time.time();gbm=HistGradientBoostingRegressor(max_iter=300,learning_rate=.08,
              max_leaf_nodes=63,early_stopping=False,random_state=0)
    gbm.fit(xtr,logy)
    family['GBM + time' if with_time else 'GBM no time']=np.exp(gbm.predict(xte))
    print('GBM fit',('with time' if with_time else 'without time'),f'{time.time()-t:.1f}s')
family_df=pd.DataFrame([{'model':name,'MAE_clean':stats(y,p,mask_family)['MAE'],
                         'MAE_all':stats(y,p,np.ones(len(y),bool))['MAE'],
                         'R2_all':stats(y,p,np.ones(len(y),bool))['R2']}
                        for name,p in family.items()])
print('REFERENCE-imputation fixed-mask family audit (unfiltered fit; distinct from selected pipeline):')
print(family_df.round(4).to_string(index=False))
fig,ax=plt.subplots(3,3,figsize=(17,12));a=ax.ravel()
q=family_df.sort_values('MAE_clean');inds=np.arange(len(q))
a[0].barh(inds-.17,q.MAE_clean,height=.33,label='clean');a[0].barh(inds+.17,q.MAE_all,height=.33,label='all')
a[0].set_yticks(inds,q.model);a[0].legend();a[0].set_title('Family MAE: clean vs ALL ($)')
res=pd.DataFrame({'date':te.date.values,'actual':y,'pred':pred})
g=res[clean_mask].groupby('date')[['actual','pred']].mean()
a[1].plot(g.index,g.actual,label='actual clean');a[1].plot(g.index,g.pred,label='predicted')
a[1].legend();a[1].set_title('Sep–Oct: actual vs selected-policy prediction')
a[2].scatter(y[clean_mask],pred[clean_mask],s=3,alpha=.16)
a[2].plot([0,y.max()],[0,y.max()],c='red',ls='--');a[2].set(xlabel='actual',ylabel='selected-policy prediction',
      xlim=(0,7500),ylim=(0,7500),title='Clean holdout: predicted vs actual')
a[3].hist(100*np.log(pred[clean_mask]/y[clean_mask]),bins=80)
a[3].set(xlabel='log error (%)',title='Clean holdout log-error distribution')
error=pd.DataFrame({'date':te.date.dt.to_period('M').astype(str).values,
                    'err':np.abs(y-pred),'clean':clean_mask})
g=error.groupby('date').err.mean();gc=error[error.clean].groupby('date').err.mean()
a[4].bar(np.arange(len(g))-.2,gc.values,width=.4,label='clean')
a[4].bar(np.arange(len(g))+.2,g.values,width=.4,label='all')
a[4].set_xticks(np.arange(len(g)),g.index);a[4].legend();a[4].set_title('MAE by holdout month')
a[5].scatter(te.distance.iloc[::8],(pred-y)[::8],s=4,alpha=.2)
a[5].axhline(0,c='red',ls='--');a[5].set(xlabel='distance (mi)',ylabel='prediction − actual ($)',
                                                 title='All-row dollar residuals (sample)')
coef=full['coef'];k=len(full['cities'])-1
labels=['intercept','log distance','log dist²','market','quote','log |weight|','reefer','flatbed','time/day']
a[6].barh(labels[::-1],coef[:9][::-1]);a[6].set_title('Final fit: first 9 coefficients')
city_effects=pd.Series(np.r_[0,coef[9:9+k]],index=full['cities'])
top=city_effects.reindex(city_effects.abs().nlargest(12).index).sort_values()
a[7].barh(top.index,100*(np.exp(top.values)-1));a[7].set_title('Pickup-city effects (%) vs baseline')
# Date coverage: compare observed label dates and supplied validation feature dates.
a[8].hist(d.date,bins=20,alpha=.65,label='labeled training')
a[8].hist(v.date,bins=8,alpha=.65,label='validation features')
a[8].legend();a[8].set_title('Date coverage: train vs validation')
fig.suptitle('02 / Out-of-time evaluation and candidate-model audit',fontsize=16)
finish(fig,'02_validation_models.png')

# ────────────────────────────────────────────────────────────────────────────
# PAGE 3 — 12K PREDICTIONS, DECEMBER CHART, FEATURE AVAILABILITY, SCORER
# ────────────────────────────────────────────────────────────────────────────
print('\nPAGE 3 — final output and December feature-availability sensitivity')
print('Validation null market_index rows filled with same-date supplied-feature medians:',
      int(v.market_index.isna().sum()))
val_pred=np.round(predict(full,v,source=v),2)
same=np.round(predict(full,dec,source=v),2)
fallback=np.round(predict(full,dec),2)
if PRED:
    saved=pd.read_csv(PRED)[['load_id','predicted_rate']]
    check=v[['load_id']].assign(recomputed=val_pred).merge(saved,on='load_id',how='left',validate='one_to_one')
    print('Validation saved-file rows:',len(saved),'| missing matching IDs:',int(check.predicted_rate.isna().sum()),
          '| max abs difference:',(check.recomputed-check.predicted_rate).abs().max())
if DEC_PRED:
    saved_dec=pd.read_csv(DEC_PRED)
    saved_dec['date']=pd.to_datetime(saved_dec['date'])
    check=dec[['date']].assign(recomputed=same).merge(saved_dec[['date','predicted_rate']],on='date',
                                                 how='left',validate='one_to_one')
    print('December saved-file rows:',len(saved_dec),'| max abs difference:',
          (check.recomputed-check.predicted_rate).abs().max())
print('December signal-source comparison:')
for name,p in [('same-date supplied validation FEATURES',same),('training medians only',fallback)]:
    print(name,'| mean',round(p.mean(),2),'| range',round(p.min(),2),'to',round(p.max(),2),
          '| peak',dec.date.iloc[np.argmax(p)].date())
print('Same-date minus training-median: mean $',round((same-fallback).mean(),2),
      '| max abs $',round(np.abs(same-fallback).max(),2))
fig,ax=plt.subplots(2,3,figsize=(17,8));a=ax.ravel()
a[0].hist(val_pred,bins=80,color='#2563eb');a[0].set(xlabel='predicted rate ($)',title='12,000 validation predictions')
a[1].plot(v.assign(pred=val_pred).groupby('date').pred.mean())
a[1].set_title('Daily mean validation prediction (no labels)')
a[2].plot(dec.date,same,'o-',ms=3,label='same-date validation features (submitted)')
a[2].plot(dec.date,fallback,'--',label='training-only medians')
a[2].legend(fontsize=8);a[2].tick_params(axis='x',rotation=35);a[2].set_title('December curve: feature-source sensitivity')
by=v.assign(_day=v.date).groupby('_day')[['market_index','quote_signal']].median()
a[3].plot(by.index,by.market_index,label='market_index')
a[3].axhline(full['m']['market'],ls='--',alpha=.6,label='train median')
a[3].legend();a[3].set_title('Supplied daily market signal')
a[4].plot(by.index,by.quote_signal,label='quote_signal',c='#d97706')
a[4].axhline(full['m']['quote'],ls='--',alpha=.6,label='train median')
a[4].legend();a[4].set_title('Supplied daily quote signal')
if CHART:
    a[5].imshow(plt.imread(CHART));a[5].axis('off');a[5].set_title('Official scorer chart (existing file)')
else:
    a[5].axis('off');a[5].text(.05,.6,'Official scorer chart not found.\nRun score.py separately to create it.',
                               transform=a[5].transAxes,fontsize=12)
fig.suptitle('03 / Deliverables and December assumptions (no hidden target scores)',fontsize=16)
finish(fig,'03_predictions_december.png')
# ────────────────────────────────────────────────────────────────────────────
# PAGE 4 — 7 ROLLING TWO-MONTH FOLDS, LANE VARIATION, NEW CITY COVERAGE
# Four time treatments mirror audit Cell 9B. Training is always before each test window.
# The label-based clean mask is used only AFTER prediction for reporting metrics.
# ────────────────────────────────────────────────────────────────────────────
print('\nPAGE 4 — reference-imputation rolling windows / city & lane checks')
roll=[];full_city_vocab=sorted(d.pickup.unique().tolist())
for month in range(3,10):
    cut=pd.Timestamp(2025,month,1); end=cut+pd.DateOffset(months=2)
    a=d[d.date<cut]; b=d[(d.date>=cut)&(d.date<end)]
    if len(a)<5000 or len(b)<100: continue
    mm=meds(a); yytr=np.log(a.posted_rate.to_numpy(float));yyte=b.posted_rate.to_numpy(float)
    tt=np.log(a.distance.to_numpy(float)); td=np.log(b.distance.to_numpy(float))
    bd=np.linalg.lstsq(np.column_stack([np.ones(len(a)),tt]),yytr,rcond=None)[0]
    core=np.abs(np.log(yyte)-np.column_stack([np.ones(len(b)),td])@bd)<.4
    recent=(a.date >= a.date.max()-pd.Timedelta(days=30)).to_numpy()
    modes={}
    for wt in [False,True]:
        Xa=X(a,mm,full_city_vocab,with_time=wt);Xb=X(b,mm,full_city_vocab,with_time=wt)
        bb=np.linalg.lstsq(Xa,yytr,rcond=None)[0]
        pp=np.exp(Xb@bb);offset=np.mean((yytr-Xa@bb)[recent])
        if wt: modes.update({'linear':pp,'linear+anchor':pp*np.exp(offset)})
        else: modes.update({'none':pp,'anchor-only':pp*np.exp(offset)})
    for label,pred_roll in modes.items():
        roll.append({'fold':f'{cut:%b}–{(end-pd.Timedelta(days=1)):%b}',
                     'mode':label,'n_clean':int(core.sum()),
                     'MAE_clean':np.mean(np.abs(yyte[core]-pred_roll[core])),
                     'bias_log':np.mean(np.log(pred_roll[core]/yyte[core]))})
roll=pd.DataFrame(roll)
print('7-fold clean MAE by time handling (measured, not a selected verdict):')
print(roll.pivot(index='fold',columns='mode',values='MAE_clean')
      .reindex(roll.loc[roll['mode']=='none','fold'].tolist()).round(1).to_string())
print('Unweighted 7-fold mean:',roll.groupby('mode').MAE_clean.mean().round(1).to_dict())
all_cities=set(d.pickup)|set(d.delivery)
new_cities=sorted((set(v.pickup)|set(v.delivery))-all_cities)
vc=pd.Series({c:int(((v.pickup==c)|(v.delivery==c)).sum()) for c in new_cities}).sort_values()
print('Unseen validation cities and affected load counts:',vc.to_dict())
lanecv=(d.groupby(['pickup','delivery']).distance.agg(['mean','std','count'])
        .query('count >= 2').assign(cv=lambda f:f['std']/f['mean']))
print('Repeated-lane distance CV median:',lanecv.cv.median())
fig,ax=plt.subplots(2,3,figsize=(17,8));z=ax.ravel()
pivot=roll.pivot(index='fold',columns='mode',values='MAE_clean')
order=roll.loc[roll['mode']=='none','fold'].tolist();pivot=pivot.reindex(order)
for name in ['none','linear','linear+anchor','anchor-only']:
    z[0].plot(pivot.index,pivot[name],marker='o',label=name)
z[0].tick_params(axis='x',rotation=40);z[0].legend(fontsize=7)
z[0].set_title('Reference-imputation rolling two-month MAE (clean rows)')
bias=roll.pivot(index='fold',columns='mode',values='bias_log').reindex(order)
for name in ['none','linear','linear+anchor','anchor-only']:
    z[1].plot(bias.index,100*bias[name],marker='o',label=name)
z[1].axhline(0,c='black',lw=.8);z[1].tick_params(axis='x',rotation=40)
z[1].set_title('Rolling log bias (%)')
counts=roll[roll['mode']=='none'].set_index('fold').reindex(order)
z[2].bar(counts.index,counts.n_clean);z[2].tick_params(axis='x',rotation=40)
z[2].set_title('Number of clean evaluation rows per window')
if len(vc): z[3].barh(vc.index,vc.values)
else: z[3].text(.1,.5,'No unseen cities',transform=z[3].transAxes)
z[3].set_title('Validation rows involving each unseen city')
z[4].hist(100*lanecv.cv.dropna(),bins=60,range=(0,20),color='#8b5cf6')
z[4].set(xlabel='coefficient of variation (%)',title='Distance variation within repeated lanes')
agg=d.groupby('date').agg(rate=('posted_rate','median'),market=('market_index','median'))
z[5].plot(agg.index,agg.rate,label='daily median rate',color='#2563eb')
z2=z[5].twinx();z2.plot(agg.index,agg.market,label='daily median market',alpha=.7,color='#d97706')
z[5].set_title('Training daily rate vs market (separate axes)')
fig.suptitle('04 / Rolling evaluation and deployment coverage',fontsize=16)
finish(fig,'04_rolling_city_lane.png')

print('\nDONE. PNG dashboards:',[str(OUT/f) for f in
    ['01_data_quality.png','02_validation_models.png','03_predictions_december.png','04_rolling_city_lane.png']])
print('All original/submitted CSVs were read, NOT modified. These charts do not test hidden accuracy.')
